# SOLiD-PINN v0.8.9 — Null-calibrated adaptive closure
CPU. Run all; authorize Drive. 38 null calibration cases + 64 independent held-out cases, three fitted models per case. No PAT, private inputs or GitHub writes. Includes weak alternatives and false activations. Synthetic only; 10% marginal null false-activation target under exchangeability, not experimental guarantee.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, tempfile, hashlib, json, math
drive.mount("/content/drive")
REV="8f665a002529b0ba29723245affc01d4887412c0"
repo=Path(tempfile.mkdtemp(prefix="solid-pinn-v089-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"adaptive_closure.py":"aef5ff03636bdd6dc09b6cf985182c37f9fa0f450f721c52dcd8b38db59cd0a5","test_adaptive_closure.py":"600def603270a124d9bdc79b0f6a68985653b22ed5bb4708d1df1aa465c05521","missing_physics.py":"4dedec06a174a7f1df04b5b147e181cb8fc743a56f88d915f7d249e629768426","dynamic_benchmark.py":"526d3ab253e77e1adf8cbc3b1dc1a56b8317e5e0d9b4210814fa9479a799fc73"}
for name,digest in hashes.items():
    assert hashlib.sha256((repo/"scripts"/name).read_bytes()).hexdigest()==digest
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_adaptive_closure.py"],cwd=repo,env=env,check=True)
out=repo/"results.json"
subprocess.run([sys.executable,"scripts/adaptive_closure.py","--out",str(out)],cwd=repo,env=env,check=True)
d=json.loads(out.read_text())
assert len(d["calibration"])==38 and len(d["test"])==64
assert {(x["noise"],x["seed"],x["truth"]) for x in d["calibration"]}=={(n,s,"null") for n in (1,5) for s in range(1000,1019)}
assert {(x["noise"],x["seed"],x["truth"]) for x in d["test"]}=={(n,s,t) for n in (1,5) for s in range(2000,2008) for t in ("null","rest","peak","weak_rest")}
for n in (1.,5.):
    q=max(0.,sorted(x["gain"] for x in d["calibration"] if x["noise"]==n)[17])
    assert d["thresholds"][str(n)]==q
    for x in (v for v in d["test"] if v["noise"]==n):
        scores=x["validation_scores"]
        best=min(("linear","hybrid_rbf"),key=scores.get)
        gain=scores["calibrated"]-scores[best]
        assert x["gain"]==gain and x["gated"]==(best if gain>q else "calibrated")
        assert set(x["models"])=={"calibrated","linear","hybrid_rbf"}
        assert all(0<=m["min_state"]<=m["max_state"]<=1 for m in x["models"].values())
assert all(x["reference_error"]<1e-6 for x in d["calibration"]+d["test"])
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=4)
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/drive/MyDrive/SOLiD-PINN/colab_runs/v089/handoff")
root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v089_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    part=dest.with_suffix(".part");part.write_bytes(raw);part.replace(dest)
assert dest.read_bytes()==raw
print("DRIVE VERIFIED",dest.name,sha)
files.download(str(dest))
